# cap10_joint_contract_bc_ppo_drift_screen_v1

Purpose: continue the cap10 joint-contract line after the BC slice gate passed by running a short walk-forward PPO diagnostic and explicitly auditing PPO-vs-BC drawdown drift.

Governance:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1` runtime composer.
- Direct comparison baseline: `cap10_joint_contract_bc_slice_screen_v1`.
- Role: `diagnose`.
- Changed layer: `model_selection` / `bc_to_ppo_handoff` only.
- This notebook runs a short diagnostic PPO screen using the existing `sparse_release_bc_ppo_screen_v1` 4-iter profile. It does not create, retain, or promote an actor bundle.
- Target gate remains excess >= +30pp and MaxDD <= 15% on full fixed split plus walk-forward windows. Passing this notebook would only permit a fuller PPO/backtest run.

External basis:
- Qlib framework separation and RL support: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- PPO policy-gradient handoff: https://arxiv.org/abs/1707.06347
- Conservative/offline-RL caution before policy improvement: https://arxiv.org/abs/2006.04779
- Tianshou PPO implementation surface used by this harness: https://tianshou.org/en/stable/03_api/policy/modelfree/ppo.html
- Backtest-overfit discipline: https://doi.org/10.21314/JCF.2016.322

Closest historical success: `cap10_joint_contract_bc_slice_screen_v1` showed the raw joint target contract can be fit by a low-complexity BC actor across both available fold-local target sets.

Closest historical failure: previous sparse-release PPO screens improved some relative-return metrics but failed the +30pp / <=15% MaxDD joint gate because PPO either drifted drawdown above target or failed recovery participation.

Why this is not a duplicate: this screen uses the new joint contract, checks release + downtrend + drawdown-control slices per fold, and blocks continuation if short PPO worsens BC by more than the drift stop rules.


In [1]:
from __future__ import annotations

import copy
import json
import math
import os
import random
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.soxl_quant.app.selection import train_with_selection
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.environment import build_initial_states
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.observation import observation_dim_for_profile
from src.soxl_quant.policy.policy_builder import build_policy
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'cap10_joint_contract_bc_ppo_drift_screen_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
SOURCE_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_joint_contract_bc_slice_screen_v1'
TARGET_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_foldlocal_joint_target_contract_v1'
TARGETS_PATH = TARGET_CARD / 'joint_target_contract.csv'
SOURCE_SELECTION_PATH = SOURCE_CARD / 'selection.json'
FRAME_PATH = ROOT / 'data' / 'soxl_profile_balanced_rule_residual_v22.pkl'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
BC_GLOBAL_ACCURACY_MIN = 0.70
BC_SLICE_ACCURACY_MIN = 0.60
REQUIRED_SLICES = {'release_gate_state': 20, 'is_downtrend_state': 100, 'drawdown_control_state': 20}
DRIFT_RELATIVE_FLOOR_PP = -5.0
DRIFT_MAX_DD_CEILING_PP = 2.0
CARD.mkdir(parents=True, exist_ok=True)

WALK_FORWARD_WINDOWS = {
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}

def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value

def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + chr(10), encoding='utf-8')

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print('root', ROOT)
print('card', CARD)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


root /Users/rziiiii/.openclaw/workspace
card /Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/cap10_joint_contract_bc_ppo_drift_screen_v1


In [2]:
source_selection = json.loads(SOURCE_SELECTION_PATH.read_text())
profile = load_profile(PROFILE_NAME)
frame = pd.read_pickle(FRAME_PATH)
targets = pd.read_csv(TARGETS_PATH)
targets['datetime'] = pd.to_datetime(targets['datetime'])
for col in ('is_rebound_cluster', 'is_downtrend_state', 'release_gate_state', 'drawdown_control_state'):
    if col in targets.columns:
        targets[col] = targets[col].fillna(False).astype(bool)

coverage_rows = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    train_start = pd.Timestamp(spec['train'][0])
    train_end = pd.Timestamp(spec['train'][1])
    valid_start = pd.Timestamp(spec['valid'][0])
    valid_end = pd.Timestamp(spec['valid'][1])
    test_start = pd.Timestamp(spec['test'][0])
    test_end = pd.Timestamp(spec['test'][1])
    fold_targets = targets[
        targets['fold'].eq(spec['target_fold'])
        & targets['datetime'].between(train_start, train_end)
    ].copy()
    all_fold_targets = targets[targets['fold'].eq(spec['target_fold'])].copy()
    coverage_rows.append({
        'fold': fold_name,
        'target_fold': spec['target_fold'],
        'all_target_rows': int(len(all_fold_targets)),
        'train_target_rows': int(len(fold_targets)),
        'all_start': None if all_fold_targets.empty else all_fold_targets['datetime'].min().date().isoformat(),
        'all_end': None if all_fold_targets.empty else all_fold_targets['datetime'].max().date().isoformat(),
        'train_start': spec['train'][0],
        'train_end': spec['train'][1],
        'valid_start': spec['valid'][0],
        'valid_end': spec['valid'][1],
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
        'valid_overlap_rows': int(targets[targets['fold'].eq(spec['target_fold']) & targets['datetime'].between(valid_start, valid_end)].shape[0]),
        'test_overlap_rows': int(targets[targets['fold'].eq(spec['target_fold']) & targets['datetime'].between(test_start, test_end)].shape[0]),
        'release_gate_rows': int(fold_targets.get('release_gate_state', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
        'downtrend_rows': int(fold_targets.get('is_downtrend_state', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
        'drawdown_control_rows': int(fold_targets.get('drawdown_control_state', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
        'rebound_rows': int(fold_targets.get('is_rebound_cluster', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
    })
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / 'coverage.csv', index=False)
write_json(CARD / 'coverage.json', {'rows': coverage.to_dict('records'), 'source_bc_decision': source_selection.get('decision')})
display(coverage)
print(json.dumps(json_safe({'source_selection': source_selection, 'coverage': coverage.to_dict('records')}), indent=2, allow_nan=False))


,fold,target_fold,all_target_rows,train_target_rows,all_start,all_end,train_start,train_end,valid_start,valid_end,test_start,test_end,valid_overlap_rows,test_overlap_rows,release_gate_rows,downtrend_rows,drawdown_control_rows,rebound_rows
0,wf_2022,wf_2022_train,505,253,2020-01-02,2021-12-31,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31,252,0,40,118,49,66
1,wf_2023_2024,wf_2023_2024_train,756,756,2020-01-02,2022-12-30,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31,0,0,40,478,49,66


{
  "source_selection": {
    "actor_retention": "no actor retained; rerun this notebook to regenerate fold-local BC actors",
    "bc_screen_gate": true,
    "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
    "changed_layer": [
      "model_selection",
      "training_objective"
    ],
    "created_at": "2026-05-25T23:15:38.660119+00:00",
    "decision": "joint_contract_bc_slice_ready_for_ppo_screen_no_promotion",
    "direct_comparison_baseline": "cap10_foldlocal_joint_target_contract_v1",
    "experiment": "cap10_joint_contract_bc_slice_screen_v1",
    "external_sources": [
      "https://arxiv.org/abs/1506.02188",
      "https://arxiv.org/abs/2006.04779",
      "https://arxiv.org/abs/2009.11189",
      "https://doi.org/10.21314/JCF.2016.322",
      "https://github.com/microsoft/qlib",
      "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551",
      "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2708678"
    ],
    "failed_checks": [],
    "next_mechan

In [3]:
def evaluate_actor_path(actor, eval_df: pd.DataFrame, price_series: pd.Series, profile, test_seg: tuple[str, str]) -> dict[str, float]:
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(
        actor,
        eval_df,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        action_levels=action_levels_for_profile(profile),
        observation_window=profile.model.observation_window,
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=profile.model.use_hybrid_overlay,
        delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None,
        feature_columns=feature_columns,
        mode_router_config=profile.mode_router,
        position_mapping=profile.model.position_mapping,
    )
    bt = backtest_profile(
        actor,
        eval_df,
        price_series,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=test_seg,
    )
    out = dict(split_metrics)
    for src, dst in [
        ('return', 'return'),
        ('sharpe', 'sharpe'),
        ('max_dd', 'max_dd'),
        ('bnh_return', 'buy_and_hold_return'),
        ('relative_outperformance', 'relative_return'),
    ]:
        if src in bt:
            out[dst] = float(bt[src])
    out['backtest_return'] = float(bt.get('return', out.get('return', 0.0)))
    out['backtest_max_dd'] = float(bt.get('max_dd', out.get('max_dd', 0.0)))
    out['backtest_relative_return'] = float(bt.get('relative_outperformance', out.get('relative_return', 0.0)))
    return out

def prefixed_metrics(prefix: str, metrics: dict[str, Any]) -> dict[str, float]:
    keys = ['return', 'sharpe', 'max_dd', 'avg_pos', 'stress_avg_pos', 'buy_and_hold_return', 'relative_return']
    return {f'{prefix}_{k}': float(metrics[k]) for k in keys if k in metrics and metrics[k] is not None}

def slice_gate_from_metrics(slice_metrics: pd.DataFrame) -> tuple[bool, dict[str, Any]]:
    indexed = slice_metrics.set_index('slice') if not slice_metrics.empty else pd.DataFrame()
    details: dict[str, Any] = {}
    gate = True
    for slice_name, min_samples in REQUIRED_SLICES.items():
        if slice_name not in indexed.index:
            row = {'samples': 0, 'accuracy': float('nan'), 'gate': False}
        else:
            item = indexed.loc[slice_name]
            row = {
                'samples': int(item.get('samples', 0)),
                'accuracy': float(item.get('accuracy', float('nan'))),
            }
            row['gate'] = bool(row['samples'] >= min_samples and math.isfinite(row['accuracy']) and row['accuracy'] >= BC_SLICE_ACCURACY_MIN)
        row['min_samples'] = int(min_samples)
        row['min_accuracy'] = float(BC_SLICE_ACCURACY_MIN)
        details[slice_name] = row
        gate = gate and bool(row['gate'])
    return bool(gate), details


In [4]:
init_qlib(QLIB_DATA_PATH)
train_config = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)
obs_dim = observation_dim_for_profile(profile)

fold_rows = []
history_rows = []
bc_slice_frames = []
drift_rules = pd.DataFrame([
    {'rule': 'target_relative_return_pp', 'threshold': TARGET_RELATIVE_RETURN, 'direction': '>='},
    {'rule': 'target_max_dd_pct', 'threshold': TARGET_MAX_DD, 'direction': '<='},
    {'rule': 'ppo_vs_bc_relative_delta_pp', 'threshold': DRIFT_RELATIVE_FLOOR_PP, 'direction': '>='},
    {'rule': 'ppo_vs_bc_max_dd_delta_pp', 'threshold': DRIFT_MAX_DD_CEILING_PP, 'direction': '<='},
    {'rule': 'bc_required_slice_accuracy', 'threshold': BC_SLICE_ACCURACY_MIN, 'direction': '>='},
])
drift_rules.to_csv(CARD / 'drift_stop_rules.csv', index=False)

for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    print(f"\n[fold:{fold_name}] target_fold={spec['target_fold']} train={spec['train']} valid={spec['valid']} test={spec['test']}")
    set_seed(SEED)
    train_start = pd.Timestamp(spec['train'][0])
    train_end = pd.Timestamp(spec['train'][1])
    valid_start = pd.Timestamp(spec['valid'][0])
    valid_end = pd.Timestamp(spec['valid'][1])
    test_start = pd.Timestamp(spec['test'][0])
    test_end = pd.Timestamp(spec['test'][1])
    fold_targets = targets[
        targets['fold'].eq(spec['target_fold'])
        & targets['datetime'].between(train_start, train_end)
    ].copy()
    valid_overlap = targets[targets['fold'].eq(spec['target_fold']) & targets['datetime'].between(valid_start, valid_end)]
    test_overlap = targets[targets['fold'].eq(spec['target_fold']) & targets['datetime'].between(test_start, test_end)]
    if fold_targets.empty:
        fold_rows.append({'fold': fold_name, 'status': 'skipped_no_train_targets'})
        continue
    if not test_overlap.empty:
        fold_rows.append({'fold': fold_name, 'status': 'blocked_test_overlap', 'test_overlap_rows': int(len(test_overlap))})
        continue

    bc_result = fit_state_objective_bc_actor_from_frame(
        frame,
        targets=fold_targets,
        profile=profile,
        train_config=train_config,
        sample_weight_config=None,
        sample_size=0,
    )
    bc_dataset = build_state_objective_bc_dataset_from_frame(frame, targets=fold_targets, profile=profile, sample_size=0)
    bc_slice_metrics = evaluate_bc_actor_slice_fit(
        bc_result.actor,
        bc_result.obs,
        bc_result.actions,
        bc_dataset.metadata,
        slice_cols=('release_gate_state', 'is_downtrend_state', 'drawdown_control_state', 'is_rebound_cluster'),
    )
    bc_slice_gate, bc_slice_details = slice_gate_from_metrics(bc_slice_metrics)
    tagged_slices = bc_slice_metrics.copy()
    tagged_slices.insert(0, 'fold', fold_name)
    bc_slice_frames.append(tagged_slices)

    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    train_df = segment_frame(dataset, 'train')
    valid_df = segment_frame(dataset, 'valid')
    test_df = segment_frame(dataset, 'test')
    price_frame = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()
    test_price_series = price_frame.loc[spec['test'][0]:spec['test'][1]]
    valid_price_series = price_frame.loc[spec['valid'][0]:spec['valid'][1]]

    train_pickle = CARD / f'{fold_name}_train_frame.pkl'
    train_df.to_pickle(train_pickle)
    initial_states = build_initial_states([(train_pickle, train_df, 0)], profile, smoke=False)
    if not initial_states:
        fold_rows.append({'fold': fold_name, 'status': 'skipped_no_initial_states'})
        continue

    policy, actor = build_policy(obs_dim, profile)
    actor.load_state_dict(copy.deepcopy(bc_result.actor.state_dict()))
    actor.eval()
    bc_test_metrics = evaluate_actor_path(actor, test_df, test_price_series, profile, spec['test'])
    bc_valid_metrics = evaluate_actor_path(actor, valid_df, valid_price_series, profile, spec['valid'])

    actor.train()
    selection = train_with_selection(
        policy=policy,
        actor=actor,
        initial_states=initial_states,
        obs_dim=obs_dim,
        profile=profile,
        valid_df=valid_df,
        smoke=False,
        test_df=None,
    )
    actor.eval()
    selected_test_metrics = evaluate_actor_path(actor, test_df, test_price_series, profile, spec['test'])
    selected_valid_metrics = selection['best_metrics']
    for item in selection.get('history', []):
        row = dict(item)
        row['fold'] = fold_name
        history_rows.append(row)

    rel_delta = float(selected_test_metrics.get('relative_return', 0.0) - bc_test_metrics.get('relative_return', 0.0))
    dd_delta = float(selected_test_metrics.get('max_dd', 0.0) - bc_test_metrics.get('max_dd', 0.0))
    selected_gate = bool(
        selected_test_metrics.get('relative_return', -1e9) >= TARGET_RELATIVE_RETURN
        and selected_test_metrics.get('max_dd', 1e9) <= TARGET_MAX_DD
    )
    bc_gate = bool(
        bc_test_metrics.get('relative_return', -1e9) >= TARGET_RELATIVE_RETURN
        and bc_test_metrics.get('max_dd', 1e9) <= TARGET_MAX_DD
    )
    drift_gate = bool(rel_delta >= DRIFT_RELATIVE_FLOOR_PP and dd_delta <= DRIFT_MAX_DD_CEILING_PP)
    ppo_continuation_gate = bool(selected_gate and drift_gate and bc_slice_gate)

    fold_rows.append({
        'fold': fold_name,
        'status': 'evaluated',
        'target_fold': spec['target_fold'],
        'train_rows': int(len(train_df)),
        'valid_rows': int(len(valid_df)),
        'test_rows': int(len(test_df)),
        'initial_states': int(len(initial_states)),
        'target_rows': int(len(fold_targets)),
        'valid_overlap_rows': int(len(valid_overlap)),
        'test_overlap_rows': int(len(test_overlap)),
        'release_gate_rows': int(fold_targets.get('release_gate_state', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
        'downtrend_rows': int(fold_targets.get('is_downtrend_state', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
        'drawdown_control_rows': int(fold_targets.get('drawdown_control_state', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum()),
        'bc_after_accuracy': float(bc_result.metrics['after_accuracy']),
        'bc_after_loss': float(bc_result.metrics['after_loss']),
        'bc_slice_gate': bc_slice_gate,
        'bc_release_accuracy': float(bc_slice_details['release_gate_state']['accuracy']),
        'bc_release_samples': int(bc_slice_details['release_gate_state']['samples']),
        'bc_downtrend_accuracy': float(bc_slice_details['is_downtrend_state']['accuracy']),
        'bc_downtrend_samples': int(bc_slice_details['is_downtrend_state']['samples']),
        'bc_drawdown_control_accuracy': float(bc_slice_details['drawdown_control_state']['accuracy']),
        'bc_drawdown_control_samples': int(bc_slice_details['drawdown_control_state']['samples']),
        'selected_valid_score': float(selection['best_score']),
        'selected_valid_return': float(selected_valid_metrics.get('return', 0.0)),
        'selected_valid_sharpe': float(selected_valid_metrics.get('sharpe', 0.0)),
        'selected_valid_max_dd': float(selected_valid_metrics.get('max_dd', 0.0)),
        'ppo_vs_bc_relative_delta': rel_delta,
        'ppo_vs_bc_max_dd_delta': dd_delta,
        'bc_gate': bc_gate,
        'selected_gate': selected_gate,
        'drift_gate': drift_gate,
        'fold_gate': ppo_continuation_gate,
        **prefixed_metrics('bc_test', bc_test_metrics),
        **prefixed_metrics('selected_test', selected_test_metrics),
        **prefixed_metrics('bc_valid', bc_valid_metrics),
    })

fold_results = pd.DataFrame(fold_rows)
history = pd.DataFrame(history_rows)
bc_slice_metrics_all = pd.concat(bc_slice_frames, ignore_index=True) if bc_slice_frames else pd.DataFrame()

fold_results.to_csv(CARD / 'ppo_fold_results.csv', index=False)
history.to_csv(CARD / 'ppo_selection_history.csv', index=False)
bc_slice_metrics_all.to_csv(CARD / 'bc_slice_metrics.csv', index=False)
for train_pickle in CARD.glob('wf_*_train_frame.pkl'):
    train_pickle.unlink(missing_ok=True)

display(fold_results)
display(history)
display(bc_slice_metrics_all)


[29307:MainThread](2026-05-26 07:26:47,229) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[29307:MainThread](2026-05-26 07:26:48,745) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[29307:MainThread](2026-05-26 07:26:48,750) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}



[fold:wf_2022] target_fold=wf_2022_train train=('2010-03-11', '2020-12-31') valid=('2021-01-01', '2021-12-31') test=('2022-01-01', '2022-12-31')


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)


,fold,status,target_fold,train_rows,valid_rows,test_rows,initial_states,target_rows,valid_overlap_rows,test_overlap_rows,...,selected_test_stress_avg_pos,selected_test_buy_and_hold_return,selected_test_relative_return,bc_valid_return,bc_valid_sharpe,bc_valid_max_dd,bc_valid_avg_pos,bc_valid_stress_avg_pos,bc_valid_buy_and_hold_return,bc_valid_relative_return
0,wf_2022,evaluated,wf_2022_train,2470,252,251,21,253,252,0,...,0.240116,-76.614257,41.770333,4.472158,0.351971,9.424797,0.330955,0.320833,78.905912,-58.602980
1,wf_2023_2024,evaluated,wf_2023_2024_train,2973,250,252,23,756,0,0,...,0.114261,-39.952710,30.501116,25.661500,1.589522,11.572674,0.272619,0.000000,79.498558,-38.468538


,iter,score,return,sharpe,max_dd,avg_pos,stress_avg_pos,stress_days,buy_and_hold_return,relative_return,fold
0,0,40.862227,20.302932,1.101080,12.185863,0.330955,0.320833,3.0,78.905912,-58.602980,wf_2022
1,4,59.824124,35.355293,1.318799,15.892958,0.463466,0.442889,3.0,78.905912,-43.550619,wf_2022
2,0,80.967029,41.030020,2.065908,11.509663,0.272619,0.000000,0.0,79.498558,-38.468538,wf_2023_2024
3,4,67.773181,31.491659,1.887831,12.292481,0.237103,0.000000,0.0,79.498558,-48.006899,wf_2023_2024


,fold,slice,samples,accuracy,loss,label_unique_actions,predicted_unique_actions,label_dominant_action,label_dominant_share,predicted_dominant_action,predicted_dominant_share
0,wf_2022,all,253,0.833992,1.073453,5,4,4,0.513834,4,0.616601
1,wf_2022,drawdown_control_state,49,0.918367,0.985184,3,2,4,0.857143,4,0.918367
2,wf_2022,is_downtrend_state,118,0.779661,1.129647,4,3,4,0.372881,4,0.533898
3,wf_2022,is_rebound_cluster,66,0.803030,1.101154,2,1,4,0.803030,4,1.000000
4,wf_2022,release_gate_state,40,1.000000,0.905056,1,1,4,1.000000,4,1.000000
5,wf_2023_2024,all,755,0.768212,1.146839,5,4,1,0.295364,1,0.368212
6,wf_2023_2024,drawdown_control_state,49,0.979592,0.929182,3,3,4,0.857143,4,0.836735
7,wf_2023_2024,is_downtrend_state,477,0.779874,1.133712,5,4,1,0.383648,1,0.459119
8,wf_2023_2024,is_rebound_cluster,66,0.924242,0.984419,2,2,4,0.803030,4,0.818182
9,wf_2023_2024,release_gate_state,40,0.950000,0.954481,1,2,4,1.000000,4,0.950000


In [5]:
evaluated = fold_results[fold_results['status'].eq('evaluated')].copy() if not fold_results.empty else pd.DataFrame()
all_fold_gate = bool((not evaluated.empty) and evaluated['fold_gate'].all() and len(evaluated) == len(WALK_FORWARD_WINDOWS))
any_selected_gate = bool((not evaluated.empty) and evaluated['selected_gate'].any())
any_bc_gate = bool((not evaluated.empty) and evaluated['bc_gate'].any())
all_slice_gate = bool((not evaluated.empty) and evaluated['bc_slice_gate'].all())
any_drift_fail = bool((not evaluated.empty) and (~evaluated['drift_gate'].astype(bool)).any())
wf_2022_failed = bool((not evaluated.empty) and ('wf_2022' in set(evaluated['fold'])) and (not bool(evaluated.set_index('fold').loc['wf_2022', 'fold_gate'])))

if all_fold_gate:
    decision = 'joint_contract_bc_ppo_drift_screen_passed_partial_wf_no_promotion'
    next_mechanism = 'run_full_fixed_split_plus_wf2020_wf2022_wf2023_2024_ppo_validation'
elif any_selected_gate and all_slice_gate and not any_drift_fail:
    decision = 'joint_contract_bc_ppo_screen_mixed_no_promotion'
    next_mechanism = 'inspect_failed_fold_gate_before_longer_training'
elif any_bc_gate and all_slice_gate:
    decision = 'joint_contract_bc_initialization_useful_ppo_not_improving_no_promotion'
    next_mechanism = 'use_bc_as_checkpoint_reference_or_add_ppo_drift_penalty'
elif any_drift_fail:
    decision = 'joint_contract_ppo_drift_failed_no_promotion'
    next_mechanism = 'revise_reward_or_checkpoint_selection_to_penalize_ppo_drawdown_drift'
else:
    decision = 'joint_contract_bc_ppo_screen_failed_no_promotion'
    next_mechanism = 'revise_joint_targets_or_state_loss_before_longer_training'

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'cap10_joint_contract_bc_slice_screen_v1',
    'role': 'diagnose',
    'profile': PROFILE_NAME,
    'changed_layer': ['model_selection', 'bc_to_ppo_handoff'],
    'promotion_allowed': False,
    'training_allowed_from_this_card': False,
    'diagnostic_ppo_training_ran': True,
    'promotion_blocker': 'This is a short diagnostic PPO screen. No actor bundle is promoted or retained; promotion still requires full fixed-split plus wf_2020/wf_2022/wf_2023_2024 validation.',
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'drift_relative_floor_pp': DRIFT_RELATIVE_FLOOR_PP,
    'drift_max_dd_ceiling_pp': DRIFT_MAX_DD_CEILING_PP,
    'folds_evaluated': evaluated['fold'].tolist() if not evaluated.empty else [],
    'fold_gate_pass_count': int(evaluated['fold_gate'].sum()) if not evaluated.empty and 'fold_gate' in evaluated else 0,
    'selected_gate_pass_count': int(evaluated['selected_gate'].sum()) if not evaluated.empty and 'selected_gate' in evaluated else 0,
    'bc_gate_pass_count': int(evaluated['bc_gate'].sum()) if not evaluated.empty and 'bc_gate' in evaluated else 0,
    'bc_slice_gate_pass_count': int(evaluated['bc_slice_gate'].sum()) if not evaluated.empty and 'bc_slice_gate' in evaluated else 0,
    'drift_gate_pass_count': int(evaluated['drift_gate'].sum()) if not evaluated.empty and 'drift_gate' in evaluated else 0,
    'wf_2022_failed': wf_2022_failed,
    'wf_2020_coverage': False,
    'actor_retention': 'no actor bundle is promoted or retained by this notebook; rerun the notebook to regenerate diagnostic states',
    'sidecars': [
        'coverage.csv',
        'coverage.json',
        'drift_stop_rules.csv',
        'bc_slice_metrics.csv',
        'ppo_selection_history.csv',
        'ppo_fold_results.csv',
        'selection.json',
        'summary.json',
    ],
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://arxiv.org/abs/1707.06347',
        'https://arxiv.org/abs/2006.04779',
        'https://tianshou.org/en/stable/03_api/policy/modelfree/ppo.html',
        'https://doi.org/10.21314/JCF.2016.322',
    ],
}
summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'selection': selection,
    'source_selection': source_selection,
    'coverage': coverage.to_dict(orient='records'),
    'fold_results': fold_results.to_dict(orient='records'),
    'drift_stop_rules': drift_rules.to_dict(orient='records'),
    'conclusion': [],
}
if evaluated.empty:
    summary['conclusion'].append('No folds were evaluated; do not continue PPO from this handoff.')
else:
    for row in evaluated.to_dict(orient='records'):
        summary['conclusion'].append(
            f"{row['fold']}: BC relative={row.get('bc_test_relative_return', float('nan')):.2f}pp / MaxDD={row.get('bc_test_max_dd', float('nan')):.2f}%; "
            f"selected relative={row.get('selected_test_relative_return', float('nan')):.2f}pp / MaxDD={row.get('selected_test_max_dd', float('nan')):.2f}%; "
            f"PPO-vs-BC relative delta={row.get('ppo_vs_bc_relative_delta', float('nan')):.2f}pp, "
            f"DD delta={row.get('ppo_vs_bc_max_dd_delta', float('nan')):.2f}pp, "
            f"selected_gate={bool(row.get('selected_gate', False))}, drift_gate={bool(row.get('drift_gate', False))}, fold_gate={bool(row.get('fold_gate', False))}."
        )
    summary['conclusion'].append(
        'No runtime promotion: this screen is shorter than full training, lacks wf_2020 coverage, and requires full fixed-split + walk-forward validation before any actor retention.'
    )
    if wf_2022_failed:
        summary['conclusion'].append(
            'wf_2022 remains a blocker for the +30pp / <=15% MaxDD objective; do not lengthen PPO without first addressing drawdown drift or checkpoint selection.'
        )

write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
print(json.dumps(json_safe(selection), indent=2, allow_nan=False))
summary


{
  "experiment": "cap10_joint_contract_bc_ppo_drift_screen_v1",
  "created_at": "2026-05-25T23:31:18.919805+00:00",
  "decision": "joint_contract_bc_initialization_useful_ppo_not_improving_no_promotion",
  "next_mechanism": "use_bc_as_checkpoint_reference_or_add_ppo_drift_penalty",
  "canonical_baseline": "v19reg_v22_budget_usage_bull_guard_v1",
  "direct_comparison_baseline": "cap10_joint_contract_bc_slice_screen_v1",
  "role": "diagnose",
  "profile": "sparse_release_bc_ppo_screen_v1",
  "changed_layer": [
    "model_selection",
    "bc_to_ppo_handoff"
  ],
  "promotion_allowed": false,
  "training_allowed_from_this_card": false,
  "diagnostic_ppo_training_ran": true,
  "promotion_blocker": "This is a short diagnostic PPO screen. No actor bundle is promoted or retained; promotion still requires full fixed-split plus wf_2020/wf_2022/wf_2023_2024 validation.",
  "target_relative_return_pp": 30.0,
  "target_max_dd_pct": 15.0,
  "drift_relative_floor_pp": -5.0,
  "drift_max_dd_ceiling_p

{'experiment': 'cap10_joint_contract_bc_ppo_drift_screen_v1',
 'decision': 'joint_contract_bc_initialization_useful_ppo_not_improving_no_promotion',
 'selection': {'experiment': 'cap10_joint_contract_bc_ppo_drift_screen_v1',
  'created_at': '2026-05-25T23:31:18.919805+00:00',
  'decision': 'joint_contract_bc_initialization_useful_ppo_not_improving_no_promotion',
  'next_mechanism': 'use_bc_as_checkpoint_reference_or_add_ppo_drift_penalty',
  'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
  'direct_comparison_baseline': 'cap10_joint_contract_bc_slice_screen_v1',
  'role': 'diagnose',
  'profile': 'sparse_release_bc_ppo_screen_v1',
  'changed_layer': ['model_selection', 'bc_to_ppo_handoff'],
  'promotion_allowed': False,
  'training_allowed_from_this_card': False,
  'diagnostic_ppo_training_ran': True,
  'promotion_blocker': 'This is a short diagnostic PPO screen. No actor bundle is promoted or retained; promotion still requires full fixed-split plus wf_2020/wf_2022/wf_20

In [6]:
selection = json.loads((CARD / 'selection.json').read_text())
fold_results = pd.read_csv(CARD / 'ppo_fold_results.csv')
evaluated = fold_results[fold_results['status'].eq('evaluated')].copy()
lines = [
    '## Conclusion',
    '',
    f"Decision: `{selection['decision']}`.",
    '',
]
if evaluated.empty:
    lines.append('No walk-forward folds were evaluated; no PPO continuation is allowed.')
else:
    lines.append('Fold diagnostics:')
    for row in evaluated.itertuples(index=False):
        lines.append(
            f"- {row.fold}: BC rel {row.bc_test_relative_return:.2f}pp / MaxDD {row.bc_test_max_dd:.2f}%; "
            f"PPO rel {row.selected_test_relative_return:.2f}pp / MaxDD {row.selected_test_max_dd:.2f}%; "
            f"delta rel {row.ppo_vs_bc_relative_delta:.2f}pp, delta MaxDD {row.ppo_vs_bc_max_dd_delta:.2f}pp, "
            f"fold_gate {bool(row.fold_gate)}"
        )
lines.extend([
    '',
    'Risk note: this notebook does not retain or promote an actor. It only decides whether the joint-contract BC handoff is safe enough for a fuller PPO/backtest run.',
    '',
    f"Next action: `{selection['next_mechanism']}`.",
    '',
    'Artifacts: `ppo_fold_results.csv`, `ppo_selection_history.csv`, `bc_slice_metrics.csv`, `drift_stop_rules.csv`, `selection.json`, `summary.json`.',
])
display(Markdown(chr(10).join(lines)))


## Conclusion

Decision: `joint_contract_bc_initialization_useful_ppo_not_improving_no_promotion`.

Fold diagnostics:
- wf_2022: BC rel 36.42pp / MaxDD 18.46%; PPO rel 41.77pp / MaxDD 37.28%; delta rel 5.35pp, delta MaxDD 18.82pp, fold_gate False
- wf_2023_2024: BC rel 30.50pp / MaxDD 13.60%; PPO rel 30.50pp / MaxDD 13.60%; delta rel 0.00pp, delta MaxDD 0.00pp, fold_gate True

Risk note: this notebook does not retain or promote an actor. It only decides whether the joint-contract BC handoff is safe enough for a fuller PPO/backtest run.

Next action: `use_bc_as_checkpoint_reference_or_add_ppo_drift_penalty`.

Artifacts: `ppo_fold_results.csv`, `ppo_selection_history.csv`, `bc_slice_metrics.csv`, `drift_stop_rules.csv`, `selection.json`, `summary.json`.